В процессе решения одной аналитической задачи очень часто замечаешь и другие задачи, связанные с исследуемым вопросом. В свое время, работая над предсказанием оттока студентов со stepik, я многое изменил в структуре своих курсов.

Тогда я нашел довольно интересную закономерность прохождения онлайн курсов. Слушатели онлайн курсов очень негативно реагируют на невозможность решить задачу, иными словами, если студент застрял на определенном шаге, то он, с высокой вероятность, вообще бросит курс, чем просто пропустит этот шаг и продолжит обучение.

Давайте найдем такой стэп, используя данные о сабмитах. Для каждого пользователя найдите такой шаг, который он не смог решить, и после этого не пытался решать другие шаги. Затем найдите id шага,  который стал финальной точкой практического обучения на курсе для максимального числа пользователей.

То есть мы исследуем следующий сценарий: человек решает стэп, не может получить правильный ответ и больше не возвращается к практическим задачам. Что это за шаг такой, который отпугнул максимальное число пользователей?

In [1]:
import pandas as pd

In [2]:
# 1 solution
data = pd.read_csv('https://stepik.org/media/attachments/course/4852/submissions_data_train.zip')
data[data.submission_status == "wrong"].groupby(['user_id', 'step_id'], as_index=False).agg({'timestamp':'max'}).step_id.value_counts().keys()[0]

31978

In [3]:
# 2 solution
data.sort_values(['user_id', 'timestamp'], ascending=False).drop_duplicates(['user_id'])\
        .query("submission_status == 'wrong'").groupby('step_id')\
        .count().sort_values('submission_status').tail(1)

,timestamp,submission_status,user_id
step_id,,,
31978,154,154,154


In [5]:
# 3 solution (not right)
data.step_id.mode()

0    31978
Name: step_id, dtype: int64

In [10]:
# 1. Шаг. Пришел к выводу, что для каждого пользователя сначала надо найти его максимальный timestamp.
# Затем, к этим данным подтянуть с помощью left join информацию по step_id и статусу step'а.
# Таким образом мы будем иметь актуальные данные по каждому пользователю на его последнюю дату нахождения на курсе.

test_data = data.groupby(['user_id'], as_index=False).agg({'timestamp':'max'}).merge(data, on=['user_id','timestamp'], how='left')

# 2. Шаг. Другими словами теперь мы хотим выяснить какой шаг стал последним для большинства пользователей,
# то есть тот шаг после которого пользователь больше не появлялся на курсе. Будем исходить из предположения,
# что пользователь не смог решить степ на котором он остановился и для этого отберем только те степы на которых статус является 'wrong'.
# Для этого удалим все положительные исходы степов, которые приходятся на крайний день нахождения на курсе.

test_data_new = test_data.drop(test_data[test_data['submission_status'] == 'correct'].index)

# 3. Шаг. Осталось посчитать количество пользователей для каждого step'а, которые отвалились на конкретном шаге.

res = test_data_new.groupby(['step_id'], as_index=False).agg({'user_id':'count'}).sort_values(by='user_id', ascending=False).head()

# Итог. Как видно из данных: наибольшее количество пользователей отвалилось именно на шаге = 31978.
# Кроме того, остальные шаги можно тоже рассмотреть и возможно перенести под звездочку в конец курса.
res

,step_id,user_id
4,31978,154
28,32812,133
11,32031,97
19,32202,92
42,33481,78


In [11]:
# Определяем время последнего визита пользователя, затем соединяем с таблицей ошибочных ответов и находим id степа с максимальным количеством юзеров:

data.groupby(["user_id"], as_index = False).agg({"timestamp":"max"}).\
merge(data.query("submission_status == 'wrong'"), on = ["user_id", "timestamp"]).\
groupby(["step_id"], as_index = False).agg({"user_id": "count"}).\
rename(columns = {"user_id":"cnt"}).sort_values(["cnt"], ascending = False).head()

,step_id,cnt
4,31978,154
28,32812,133
11,32031,97
19,32202,92
42,33481,78
